# 🏔️ OE Data Intelligence — Medallion Pipeline

**Full end-to-end pipeline from raw sys logs → Gold KPI tables**

| Source | Log Format | Issues Simulated |
|---|---|---|
| **Aternity** | JSONL | Teams/Outlook degradation, high response times |
| **NetScout** | CSV | WAN packet loss, congestion, hardware faults |
| **Intune** | JSONL | Non-compliant devices, Win7/8, MDM check-in failures |
| **Infoblox** | Syslog | DNS latency, NXDOMAIN, suspicious queries |
| **Salesforce** | CSV | P1/P2 incidents, SLA breaches, recurring issues |
| **ScienceLogic** | JSONL | Infrastructure alerts, threshold breaches |

## Step 1 — Generate Mock Log Files

In [7]:
import subprocess
result = subprocess.run(
    ['python', '/home/jovyan/scripts/generate_mock_logs.py', '--days', '3'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


STDERR: python: can't open file '/home/jovyan/scripts/generate_mock_logs.py': [Errno 2] No such file or directory



## Step 2 — Preview Raw Logs

In [8]:
import json
from pathlib import Path

# Preview Aternity log
print('=== Aternity (JSONL sample) ===')
lines = Path('/home/jovyan/mock-logs/aternity').glob('*.jsonl')
for f in lines:
    with open(f) as fp:
        for i, line in enumerate(fp):
            if i < 3:
                print(json.dumps(json.loads(line), indent=2))
    break

=== Aternity (JSONL sample) ===


In [9]:
# Preview Infoblox syslog
print('=== Infoblox DNS Syslog (first 5 lines) ===')
logs = list(Path('/home/jovyan/mock-logs/infoblox').glob('*.log'))
if logs:
    with open(logs[0]) as f:
        for i, line in enumerate(f):
            if i < 5: print(line.strip())

=== Infoblox DNS Syslog (first 5 lines) ===


## Step 3 — Initialize Spark + Run Full Pipeline

In [10]:
import sys
sys.path.insert(0, '/home/jovyan/src')
sys.path.insert(0, '/home/jovyan')

from spark_session import get_spark, Paths

spark = get_spark('OE-Pipeline-Notebook')
print('Spark ready:', spark.version)

Spark ready: 3.5.0


In [11]:
from pipelines.run_pipeline import run
run()

  OE DATA INTELLIGENCE PLATFORM — FULL MEDALLION PIPELINE
  Sources: Aternity · NetScout · Intune · Infoblox ·
           Salesforce · ScienceLogic

──────────────────────────────────────────────────
  🥉  BRONZE — Raw Log Ingestion
──────────────────────────────────────────────────

📥 BRONZE: Aternity (App Performance)...

❌ PIPELINE FAILED: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/mock-logs/aternity/aternity_performance_20260510.jsonl.


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/mock-logs/aternity/aternity_performance_20260510.jsonl.

## Step 4 — Query Gold Tables (what the API serves)

In [ ]:
# App Health — show degraded/critical apps
print('=== App Health Summary (worst first) ===')
spark.read.format('delta').load(Paths.gold('app_health_summary')) \
    .select('app_name','health_status','experience_score',
            'avg_response_time_ms','crash_rate','active_user_count') \
    .orderBy('experience_score') \
    .show(10, truncate=False)

In [ ]:
# Network Root Cause Analysis
print('=== Packet Loss Root Cause Analysis ===')
spark.read.format('delta').load(Paths.gold('packet_loss_root_cause')) \
    .select('segment_name','root_cause','packet_loss_rate',
            'confidence_score','severity','recommendation') \
    .orderBy('packet_loss_rate', ascending=False) \
    .show(truncate=False)

In [ ]:
# Device Compliance
print('=== Non-Compliant Devices ===')
spark.read.format('delta').load(Paths.gold('device_health_summary')) \
    .filter('compliance_state = "NON_COMPLIANT"') \
    .select('device_name','os_name','os_version','compliance_state',
            'days_since_checkin','assigned_user','location') \
    .orderBy('days_since_checkin', ascending=False) \
    .show(10, truncate=False)

In [ ]:
# DNS Issues
print('=== DNS Server Metrics ===')
spark.read.format('delta').load(Paths.gold('dns_metrics')) \
    .select('server','total_queries','failed_queries',
            'failure_rate','avg_response_ms','nxdomain_count') \
    .orderBy('avg_response_ms', ascending=False) \
    .show(truncate=False)

In [ ]:
# Open P1/P2 Issues
print('=== Critical Open Issues ===')
spark.read.format('delta').load(Paths.gold('top_issues_summary')) \
    .filter('severity IN ("P1","P2") AND status != "RESOLVED"') \
    .select('issue_id','severity','title','category','assigned_team') \
    .orderBy('severity') \
    .show(truncate=False)

In [ ]:
# Ingestion Status
print('=== Data Source Ingestion Status ===')
spark.read.format('delta').load(Paths.gold('data_source_ingestion_status')) \
    .select('source_name','status','records_last_batch',
            'data_quality_score','latency_ms','error_message') \
    .show(truncate=False)